# AtomAgents × LAMMPS — MD demo notebook

Runs the molecular-dynamics benchmarks — **twisted CNT bundle** (the
historical variant of README "MD demo 1") or **polycrystalline Fe
compression** (README "MD demo 2") — through the full AtomAgents multi-agent
stack, building every architectural piece by hand so you can see it —
services → tools → memory → agents → graph → streamed run → artefacts.

**Prerequisites** (both from the repo root):
- MCP tool service running: `make mcp`
- `.env` filled in (LLM endpoint + key) — loaded automatically below

`lmp_mpi`/`mpirun` must be on the MCP service's PATH (they are on this
machine). The backend (`:8002`) and frontend are **not** needed — this
notebook is the script path.


## Architecture in one picture

AtomAgents is three layers. This notebook builds each one by hand, bottom-up,
so you can see exactly where every piece lives — then runs a real demo through it.

```mermaid
flowchart LR
    subgraph OUTER["Outer LangGraph — built in this notebook"]
        direction LR
        START(("start")) --> SUP["Supervisor"]
        SUP <--> MOD["Modeler"]
        SUP <--> SIM["Simulator"]
        MOD <--> SIM
        SUP --> DONE(("END"))
        MOD --> DONE
        SIM --> DONE
    end
    subgraph INNER["Inside every agent — its compiled subgraph"]
        direction LR
        TH["think"] --> PL["plan"]
        PL -- "act" --> AC["act"]
        AC -- "tool_calls" --> TN["tool / tools<br/>(MCPToolNode)"] --> AC
        AC -- "no tool calls" --> PL
        PL -- "report" --> IEND(("END"))
    end
    IEND --> RP["report() in _run_epilogue<br/>→ ReportMessage + send_to"]
    subgraph SVC["Shared services"]
        MCP["MCP tool service :9999<br/>write/exec jail runs/jobs + sandbox-exec"]
        MEM[("SQLiteRAGMemory<br/>memory.db per session")]
        LLM["LLM<br/>OpenAI-compatible or Anthropic endpoint"]
    end
    TN -.-> MCP
    INNER -.-> MEM
    INNER -.-> LLM
```

(The Supervisor prepends a `triage` node before `think` — converse / clarify /
dispatch — and can short-circuit a conversational turn without entering the
full loop. In production a fourth `Human` node exists for human-in-the-loop;
this notebook builds the unattended graph without it.)

**The three ideas that carry the whole framework:**

1. **Routing is data, not code.** The shared graph state is just `messages` +
   `send_to`. Each agent ends its turn with a report whose `send_to` field is
   a `Literal[...]` built from its `Collages` (colleague) list — schema
   pressure that makes the model pick a legal target. That is the first line
   of defence, not a guarantee: a normalizer maps sloppy targets
   (`"**Next → Simulator**"`, `"done"`) onto canonical names, and a route
   guard in `report()` coerces self-handoffs and off-list targets to `END`.
   The conditional edges below then dispatch on the resulting string.
   (On short-circuit triage turns the Supervisor skips the report LLM call
   entirely and the `send_to` is hardcoded.)
2. **Every agent is itself a graph.** All three share `BaseAgent`'s
   `think → plan ⇄ act ⇄ tools` skeleton, compiled per agent. Note the real
   topology: `act` never exits the subgraph — only `plan` can end it, and the
   hand-off report is produced *after* the subgraph returns
   (`_run_epilogue`). The Supervisor subclasses the skeleton to add `triage`
   (and an OKR `update_memory` stage); Modeler and Simulator are thin
   subclasses whose per-stage instructions differ — partly in the YAML
   templates, partly hardcoded in their `plan()`.
3. **Execution is a separate, sandboxed service.** Agents never spawn
   subprocesses, and every simulation artefact they author is written through
   MCP tools on `:9999` — where write/exec paths must resolve inside the
   `runs/jobs` jail and every child process runs under a macOS `sandbox-exec`
   profile (kernel-enforced write confinement). The framework itself writes
   only session bookkeeping in-process: the `README.md` board and
   `memory.db`, both inside the session dir. Reads and network stay open by
   design (documented gaps).


In [ ]:
# ── Environment bootstrap ────────────────────────────────────────────────
# The repo has no installed package and never calls load_dotenv — scripts
# rely on the caller's shell. A notebook kernel can start from anywhere, so
# locate the repo, extend sys.path, and load .env ourselves (values are
# never printed).
import os, sys, socket
from pathlib import Path
from urllib.parse import urlparse

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "AtomAgents").is_dir()), None)
if REPO is None:
    raise RuntimeError("Could not find the repo root (a dir containing "
                       "AtomAgents/) above the kernel CWD — launch jupyter "
                       "from inside the repo.")
for p in (str(REPO), str(REPO / "examples")):   # AtomAgents + demos packages
    if p not in sys.path:
        sys.path.insert(0, p)

def load_dotenv_min(path):
    loaded, kept = [], []
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.removeprefix("export ").partition("=")
            key, val = key.strip(), val.strip()
            if len(val) >= 2 and val[0] in "\"'" and val[-1] == val[0]:
                val = val[1:-1]               # matched quote pair
            else:
                val = val.split(" #", 1)[0].rstrip()   # unquoted inline comment
            if not val:
                continue                       # empty value would defeat defaults
            (kept if key in os.environ else loaded).append(key)
            os.environ.setdefault(key, val)
    return loaded, kept

loaded, kept = load_dotenv_min(REPO / ".env")
print(f"repo: {REPO}")
print(f".env keys loaded: {loaded}")
if kept:
    print(f"already set in the kernel env (kept): {kept}")

# Notebook-specific knobs (AFTER .env so an explicit .env value wins; BEFORE
# any AtomAgents import — config reads the environment at import time):
# - TOKEN_STREAM off: the per-token bridge is for the backend's SSE stream;
#   under stream_mode="updates" every token would be produced and discarded.
# - MAX_HANDOFFS raised: production caps at 12 agent runs per user turn and
#   relies on the user replying "Continue"; this notebook is single-turn.
os.environ.setdefault("ATOMAGENTS_TOKEN_STREAM", "0")
os.environ.setdefault("ATOMAGENTS_MAX_HANDOFFS", "40")

# ── Model backend switch ─────────────────────────────────────────────────
# Both demo notebooks run on either backend. Pick one HERE (or export
# ATOMAGENTS_LLM_PROVIDER before launching jupyter — the env var wins, so
# headless/CI runs switch backends without editing the notebook):
#   "openai"    → OpenAI-compatible route, model = ATOMAGENTS_OPENAI_MODEL (e.g. gpt-5.6-sol)
#   "anthropic" → Anthropic route,        model = ATOMAGENTS_ANTHROPIC_MODEL (e.g. claude-opus-5)
PROVIDER = os.environ.get("ATOMAGENTS_LLM_PROVIDER") or "openai"

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set — SQLiteRAGMemory's embedding "
                       "client needs it regardless of the LLM backend (and the "
                       "openai backend's LLM client too). Fill in .env at the "
                       "repo root.")
if PROVIDER == "anthropic" and not (
    os.environ.get("ANTHROPIC_AUTH_TOKEN") or os.environ.get("ANTHROPIC_API_KEY")
):
    raise RuntimeError("PROVIDER = 'anthropic' but neither ANTHROPIC_AUTH_TOKEN "
                       "nor ANTHROPIC_API_KEY is set — fill in .env at the repo "
                       "root.")

MCP_URL = os.environ.get("ATOMAGENTS_MCP_URL", "http://localhost:9999/mcp/")
u = urlparse(MCP_URL)
mcp_port = u.port or (443 if u.scheme == "https" else 80)
with socket.socket() as s:
    s.settimeout(1.0)
    if s.connect_ex((u.hostname or "127.0.0.1", mcp_port)) != 0:
        raise RuntimeError(
            f"MCP tool service is not reachable at {MCP_URL}.\n"
            "Start it from the repo root with:   make mcp\n"
            "(the service refuses to start without a workspace jail — the "
            "Makefile target sets ATOMAGENTS_TOOL_ROOT=runs/jobs for you)")
print(f"MCP service: {MCP_URL} — port open")
if PROVIDER == "anthropic":
    print(f"backend: anthropic — model: "
          f"{os.environ.get('ATOMAGENTS_ANTHROPIC_MODEL', 'claude-sonnet-5 (registry default)')}"
          f"  via  {os.environ.get('ANTHROPIC_BASE_URL', 'https://api.anthropic.com')}")
else:
    print(f"backend: {PROVIDER} — model: "
          f"{os.environ.get('ATOMAGENTS_OPENAI_MODEL', 'gpt-5 (agent default)')}"
          f"  via  {os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')}")


## 1 · Tools: the MCP service

Agents never spawn subprocesses — execution and simulation artefacts go
through MCP tools served on `:9999`. The service enforces a **write/exec
jail** (`runs/jobs`): every write path, script and working directory must
resolve inside it, and every child process runs under a macOS `sandbox-exec`
profile so even a command that slips past the guardrails cannot write outside
the jail — the kernel denies it. Reads keep only a system-directory denylist
and the network is open: both deliberate, both documented gaps.

The client comes from the repo's own `make_mcp_client()` — **not** a bare
connection dict — because it wires in the production timeouts
(`sse_read_timeout=3600s` must exceed the longest tool budget; `lammps_run`
may legitimately run 3000s, while the adapter's default read timeout is only
300s and would kill the call client-side).


In [ ]:
# ── Connect to the MCP tool service and discover tools ───────────────────
from AtomAgents.config import make_mcp_client, mcp_server_config

conn = mcp_server_config()["AtomAgent_Tools"]
print(f"url: {conn['url']}")
print(f"timeout: {conn['timeout'].total_seconds():.0f}s   "
      f"sse_read_timeout: {conn['sse_read_timeout'].total_seconds():.0f}s")

client = make_mcp_client()
tools = await client.get_tools()          # top-level await: fine in Jupyter

print(f"\n{len(tools)} tools\n")
for t in sorted(tools, key=lambda t: t.name):
    desc = ((t.description or "").strip().splitlines() or [""])[0]
    print(f"  {t.name:<24} {desc[:90]}")


## 2 · Session workspace + RAG memory

One directory per run, holding everything: the artefacts the agents author
through MCP tools, the `README.md` board the framework keeps updated
in-process, and `memory.db` (SQLite: each agent's own messages plus the
user's turns, an FTS index, task/artefact tables, embedding-based recall —
tool payloads are deliberately *not* persisted there). It **must live inside
the MCP jail** (`runs/jobs`) or every write/exec tool call would be refused.


In [ ]:
# ── Create the session workspace + memory ────────────────────────────────
import logging
from datetime import datetime
from AtomAgents.memory.sqlite_rag_memory import SQLiteRAGMemory
from AtomAgents.utils.logger import configure_agent_logger

configure_agent_logger(verbose=False)
# INFO would log the FULL inputs dict on every agent turn (~1 MB per turn
# when an image is attached) straight into the cell output — the streamed
# updates below already show everything we need.
logging.getLogger("AtomAgents").setLevel(logging.WARNING)

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
project_root = (REPO / "runs" / "jobs" / f"session-nb-md-{stamp}").resolve()
project_root.mkdir(parents=True, exist_ok=True)

memory = SQLiteRAGMemory(
    sqlite_path=str(project_root / "memory.db"),
    project_root=str(project_root),
)
print(f"session: {project_root}")


## 3 · The three agents

Each agent is one LLM + one compiled LangGraph subgraph (see the diagram
above). Things to notice in the construction below:

- **`Collages`** — the colleague names an agent may hand off to. This list
  becomes the `Literal[...]` in the structured report schema; keep `"END"`
  in it, or the agent can never *choose* to stop (the handoff-budget
  backstop in the graph wiring would still force an end).
- **`llm_config`** — passed explicitly so the model choice is visible here
  (the class default hardcodes `gpt-5` and ignores `ATOMAGENTS_MODEL`).
- **OKR tools** — `build_okr_tools` returns *in-process* tools (task board,
  artefact registry, metrics) bound to this session's memory + dir;
  production always appends them to the MCP tools, and the prompts reference
  them, so we do the same.
- **`prompt_loader`** — omitted here, so the default LAMMPS/OVITO templates load (`AtomAgents/templates/prompts/`). The FEM notebook passes the `templates/fenicsx` dir instead.
  (Careful: `prompt_loader=None` explicitly would *defeat* the default and
  break the agent — pass a real loader or omit the kwarg.)


In [ ]:
# ── Construct Supervisor, Modeler, Simulator ─────────────────────────────
from AtomAgents.agents.simulation_manager import Supervisor
from AtomAgents.agents.model_building_agent import Modeler
from AtomAgents.agents.simulator_agent import SimulatorAgent
from AtomAgents.utils.okr_tools import build_okr_tools
# prompt_loader is deliberately OMITTED below → the default LAMMPS/OVITO templates
# (AtomAgents/templates/prompts/) load via the field's default_factory.

# LLM config for the backend chosen in the bootstrap cell (PROVIDER). NOTE:
# claude models must run via the anthropic provider — routed through the
# OpenAI-compatible surface they return fenced/prose JSON that breaks the
# structured-output contract (triage/plan/report), verified live; llm_factory
# fails fast on that misconfiguration with a ValueError naming the fix.
if PROVIDER == "anthropic":
    default_model = os.environ.get("ATOMAGENTS_ANTHROPIC_MODEL", "claude-opus-5")
elif PROVIDER == "local":
    default_model = os.environ.get("ATOMAGENTS_LOCAL_MODEL", "qwen2.5")
else:
    default_model = os.environ.get("ATOMAGENTS_OPENAI_MODEL", "gpt-5")

llm_config = {
    "model": os.environ.get("ATOMAGENTS_MODEL", default_model),
    "temperature": 1.0,
    "max_tokens": 16000,
    "reasoning_effort": os.environ.get("ATOMAGENTS_REASONING_EFFORT", "medium"),
    "provider": PROVIDER,
}

# MCP tools (sandboxed service) + OKR/board tools (in-process, this session)
all_tools = tools + build_okr_tools(memory, str(project_root))

common = dict(
    memory=memory,
    tools=all_tools,
    llm_config=llm_config,
    project_root=str(project_root),
    verbose=False,
)

# Inner-subgraph ceiling 50 matches production. It is deliberately ABOVE what
# ATOMAGENTS_MAX_ACT_CYCLES (15) permits, so the act-cycle guard trips first
# and the agent wraps up gracefully instead of raising GraphRecursionError.
inner = {"recursion_limit": 50}
supervisor = Supervisor(Collages=["Modeler", "Simulator", "END"],
                        enable_okr_table=True, config=inner, **common)
modeler    = Modeler(Collages=["Supervisor", "Simulator", "END"],
                     config=inner, **common)
simulator  = SimulatorAgent(Collages=["Supervisor", "Modeler", "END"],
                            config=inner, **common)
print("agents ready:", supervisor.agent_type, modeler.agent_type, simulator.agent_type)
print("LLM:", llm_config["model"], "| reasoning_effort:", llm_config["reasoning_effort"])
print(f"tools: {len(tools)} MCP + {len(all_tools) - len(tools)} in-process OKR")


In [ ]:
# ── What one agent looks like inside ─────────────────────────────────────
# The Supervisor's compiled inner subgraph. Note there is no `report` node:
# `plan`'s "report" branch maps to END, and the hand-off report is produced
# AFTER the subgraph returns (base_agent._run_epilogue → report()).
# PNG rendering calls the public mermaid.ink API (needs network) — offline
# we fall back to the mermaid source text after ~10s.
from IPython.display import Image, display

def show_graph(compiled, title):
    g = compiled.get_graph()
    print(title)
    try:
        display(Image(g.draw_mermaid_png(max_retries=0)))
    except Exception as e:
        print(f"[png render unavailable: {type(e).__name__}] mermaid source:\n")
        print(g.draw_mermaid())

show_graph(supervisor.SubGraph,
           "Supervisor inner subgraph — triage in front; report happens after END")


## 4 · The outer multi-agent graph

The shared state is deliberately tiny — two keys:

- `messages` carries the conversation. The **`add_messages` reducer**
  id-merges whatever a node returns (full list or delta — no duplicates
  either way). Two distinct mechanisms cooperate here: the reducer exists
  because the prebuilt tool node returns *only* its new `ToolMessage` (under
  replace semantics every tool call used to wipe the history), and,
  separately, each agent's `_run_epilogue` filters its return down to the
  messages that weren't present on entry, so turns don't replay history.
- `send_to` is the routing target from each agent's report.

The path function is the production `handoff_router`: it honors `send_to`
but forces `END` once the per-turn handoff budget (`ATOMAGENTS_MAX_HANDOFFS`)
is spent — the structural backstop behind the friendlier in-report budget
check. The `recursion_limit` passed to `stream(...)` is the outer
LangGraph ceiling, *not* the effective budget — `MAX_HANDOFFS` is (which is
why the bootstrap raised it for this single-turn notebook).

`set_node_defaults` below is also straight from production: if an agent node
raises (LLM/proxy failure, inner-graph overrun), it is retried up to
`ATOMAGENTS_NODE_MAX_ATTEMPTS` times and then `agent_error_handler` ends the
turn with a visible `[node-error]` message instead of an exception exploding
out of the stream loop. The budget hierarchy, innermost first:
`MAX_ACT_CYCLES` (15 act⇄tool rounds, graceful wrap-up) < inner
`recursion_limit` (50, hard) < `MAX_HANDOFFS` (agent runs per user turn) <
outer `recursion_limit`.


In [ ]:
# ── Wire the outer graph ─────────────────────────────────────────────────
from functools import partial
from langgraph.graph import StateGraph, END
from langgraph.types import RetryPolicy
import AtomAgents.config as atom_config
from AtomAgents.utils.routing import agent_error_handler, handoff_router
from AtomAgents.workflows.state_definitions import State

def run_supervisor(state: dict): return supervisor.run(state)
def run_modeler(state: dict):    return modeler.run(state)
def run_simulator(state: dict):  return simulator.run(state)

graph = StateGraph(state_schema=State)
graph.add_node("Supervisor", run_supervisor)
graph.add_node("Modeler", run_modeler)
graph.add_node("Simulator", run_simulator)
graph.set_entry_point("Supervisor")

# Production resilience: bounded node retries, then a graceful END with a
# visible error message instead of an exception out of the stream loop.
graph.set_node_defaults(
    retry_policy=RetryPolicy(max_attempts=atom_config.NODE_MAX_ATTEMPTS),
    error_handler=agent_error_handler,
)

graph.add_conditional_edges("Supervisor", partial(handoff_router, agent="Supervisor"),
    {"Modeler": "Modeler", "Simulator": "Simulator", "END": END})
graph.add_conditional_edges("Modeler", partial(handoff_router, agent="Modeler"),
    {"Supervisor": "Supervisor", "Simulator": "Simulator", "END": END})
graph.add_conditional_edges("Simulator", partial(handoff_router, agent="Simulator"),
    {"Supervisor": "Supervisor", "Modeler": "Modeler", "END": END})

app = graph.compile()
show_graph(app, "Outer graph: Supervisor ⇄ Modeler ⇄ Simulator")


## 5 · Choose the task

- **`full`** — a benchmark prompt from `examples/demos/demos.py`:
  - `DEMO_CHOICE = "cnt"` → twisted CNT fiber bundle under tension
    (`demos.demo5`), with the reference figure attached as an image — a
    **multimodal** prompt. Note: this is the *historical variant* of the
    README's MD demo 1 (no grip bullet, plus a random-defects bullet), as the
    README's "Historical variants" note documents — not byte-identical.
  - `DEMO_CHOICE = "fe"` → polycrystalline bcc Fe under uniaxial compression
    (`demos.demo6`, byte-identical to the README's MD demo 2 prompt).
  **Expect a long run** (model building + MD + post-processing).
- **`quick`** — a minimal bounded LJ task that still exercises every
  architecture piece (triage, handoffs, MCP tools incl. the sandboxed LAMMPS
  runner, memory, board) in minutes.


In [ ]:
RUN_MODE = "full"      # "full" | "quick"
DEMO_CHOICE = "cnt"    # "cnt" → twisted CNT bundle   |   "fe" → polycrystalline Fe


In [ ]:
# ── Build the task message ───────────────────────────────────────────────
from base64 import b64encode
from langchain_core.messages import HumanMessage
import demos.demos as demos               # examples/demos/demos.py

QUICK_TASK = """Quick architecture smoke test (keep it MINIMAL):
1) Modeler: build a small Lennard-Jones fcc crystal in LJ units (lattice fcc
   0.8442, 4x4x4 box, single atom type, mass 1.0) and save it as
   model_quick.data in the project root (a short LAMMPS run ending in
   write_data is fine).
2) Simulator: write run_quick.in that reads model_quick.data, does
   velocity create at T=0.5, runs 200 NVE steps with thermo every 50; run it
   with the LAMMPS tool and save the final total energy to quick_summary.txt.
3) Supervisor: verify model_quick.data and quick_summary.txt exist, then END.
All files stay in the project root. No plots, no defects, no twisting."""

if RUN_MODE not in {"full", "quick"}:
    raise ValueError(f"unknown RUN_MODE: {RUN_MODE!r}")

img_path = None
if RUN_MODE == "quick":
    TASK = QUICK_TASK
elif DEMO_CHOICE == "cnt":
    TASK = demos.demo5   # historical MD demo 1 variant (no-grip + random defects)
    img_path = REPO / "examples" / "demos" / "demo_img" / "demo5" / "demo5.png"
elif DEMO_CHOICE == "fe":
    TASK = demos.demo6   # byte-identical to README "MD demo 2"
else:
    raise ValueError(f"unknown DEMO_CHOICE: {DEMO_CHOICE!r}")

content = TASK
if img_path is not None:
    img_b64 = b64encode(img_path.read_bytes()).decode()
    content = [{"type": "text", "text": TASK},
               {"type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_b64}"}}]
input_msg = HumanMessage(content=content)
print(TASK[:600] + ("…" if len(TASK) > 600 else ""))
if img_path is not None:
    print(f"\n[+ attached reference image: {img_path.relative_to(REPO)}]")


## 6 · Run it — streaming node-by-node

`stream_mode="updates"` yields one entry per finished node: which agent ran,
the new messages it produced (tool calls, tool results, and its report last),
and where it routed next. Long payloads (data files, scripts) are truncated
for display — agent prose is persisted in `memory.db`, and the artefacts
themselves land on disk in the session dir.

> ⚠️ **This cell blocks the kernel and *Interrupt Kernel* is inert while it
> runs** (the sync loop occupies the event-loop thread ipykernel needs to
> deliver the interrupt). If you must stop a run, *Restart Kernel* — and be
> aware any in-flight MCP-side job (e.g. a LAMMPS run) keeps running in the
> service until it finishes or times out.


In [ ]:
# ── Stream the run ───────────────────────────────────────────────────────
import time, textwrap

def show_update(update, max_chars=1600):
    for node, delta in update.items():
        if not isinstance(delta, dict):
            continue
        print(f"\n{'═'*100}\n█ {node}  →  send_to: {delta.get('send_to')}\n{'═'*100}")
        for m in delta.get("messages", []):
            name = getattr(m, "name", None)
            tool_calls = [t.get("name", "?") for t in (getattr(m, "tool_calls", None) or [])]
            head = f"── {type(m).__name__}" + (f" [{name}]" if name else "")
            if tool_calls:
                head += f"  tool_calls={tool_calls}"
            print(head)
            body = m.content if isinstance(m.content, str) else str(m.content)
            if len(body) > max_chars:
                body = body[:max_chars] + f"\n… [+{len(body)-max_chars:,} chars truncated]"
            print(textwrap.indent(body, "    "))

t0 = time.time()
final_route = None
for update in app.stream({"messages": [input_msg]},
                         config={"recursion_limit": 100},   # LangGraph ceiling;
                         stream_mode="updates"):            # MAX_HANDOFFS is the real budget
    show_update(update)
    for delta in update.values():
        if isinstance(delta, dict):
            final_route = delta.get("send_to") or final_route
print(f"\nfinished in {time.time()-t0:,.0f}s — final send_to: {final_route}")


## 7 · Inspect what the run produced

Three views of the same session: the files on disk, the `README.md` board the
framework maintained while the agents worked, and the message history
persisted by the RAG memory (each agent's own prose + the user's turns).


In [ ]:
# ── Session artefacts ────────────────────────────────────────────────────
import sqlite3
from contextlib import closing
from IPython.display import Image, display

files = sorted(p for p in project_root.rglob("*") if p.is_file())
for p in files:
    print(f"{p.stat().st_size:>12,}  {p.relative_to(project_root)}")

board = project_root / "README.md"
if board.exists():
    print("\n" + "─"*100 + "\nREADME board (maintained by the framework)\n" + "─"*100)
    print(board.read_text()[:4000])

with closing(sqlite3.connect(project_root / "memory.db")) as db:
    rows = db.execute(
        "SELECT agent_name, COUNT(*) FROM messages GROUP BY agent_name").fetchall()
print("\nmessages persisted per agent:", dict(rows))

pngs = [p for p in files if p.suffix == ".png"]
for png in pngs[:6]:
    if png.stat().st_size > 2_000_000:
        print(f"\n{png.relative_to(project_root)}: skipped inline ({png.stat().st_size:,} B)")
        continue
    print(f"\n{png.relative_to(project_root)}:")
    display(Image(str(png)))
if len(pngs) > 6:
    print(f"\n(+{len(pngs)-6} more PNGs on disk, not rendered inline)")


## How this notebook maps to the production stack

Everything here is the *script path* — the same classes the production serving
path uses, assembled by hand. When you talk to the Gradio UI, this is what runs:

| This notebook | Production counterpart |
|---|---|
| `.env` bootstrap | `AtomAgents/config.py` provider registry (`discover_providers`) |
| `make_mcp_client()` | the same function — `AtomAgents/config.py` (timeouts wired in) |
| session dir under `runs/jobs` | frontend mints `session-<model>-<conv_id>`; the backend resolves it under `runs/jobs` (default `backend-<model>-<ts>`) |
| agent construction + OKR tools | `backend/openai_backend.py` `_assemble_graph` |
| outer `StateGraph` + `handoff_router` | `backend/openai_backend.py` `_assemble_graph` (same wiring) |
| `app.stream(...)` loop | backend `/v1/chat/completions` streaming (+ HITL resume, token bridge) |

What production adds that this notebook deliberately omits for legibility:
a `SqliteSaver` checkpointer (multi-turn continuity — this notebook is
single-turn, so state lives only in the `stream` call) and the `Human` node.
The `interrupt()`-based Human node is available on the
script path too (`SimulationAgentSystem(with_human=True)`), but only the
backend has the *resume* half — detecting a pending interrupt and streaming
`Command(resume=…)` — so this notebook runs unattended.

One streaming detail worth knowing: prose LLM calls are tagged so the backend
can stream them token-by-token; structured calls (plan/report/triage) stay
untagged and non-streaming — that tag gate is what keeps raw schema JSON out
of the user-facing token stream. Production registers the async `arun` as the
node callable; this notebook uses the sync `run`, which hops to a worker
thread under Jupyter's event loop.

(`AtomAgents/core/simulation_agent_system.py` is a *third* assembly of the
same pieces — a standalone job manager (`submit_job`) used by tests — not the
backend serving path.)


## Where to go next

- **Production stack**: `make backend` + `make frontend` serve this same graph
  behind an OpenAI-compatible API with session management, token streaming and
  human-in-the-loop — see the README's *Run the stack* section.
- **Benchmark evidence**: `analysis/reference_comparison/` holds the
  agent-vs-reference overlay scripts (`compare_demo1.py`, `compare_demo2.py`)
  used for the paper figures.
- **The FEM side**: `examples/notebooks/demo_fenicsx.ipynb` is this same
  walkthrough with the `templates/fenicsx` prompts and the plate-with-hole
  benchmark.
